# The Elder Scrolls Alchemy Parser v2.0

This is an advanced and modified version of [TES Alchemy Parser_v1.0](https://github.com/ed-cybros/tes-alchemy-parser-v1.0) - a fan project initially intended for personal use within The Elder Scrolls games series role-playing community with the purpose of collecting and normalizing  the games' alchemy data from [The Unoffscial Elder Scrolls Pages website](https://en.uesp.net/wiki/Lore:Alchemy).

# 1. Project Overview

## Project Goal

Alchemy effects are represented differently across multiple Elder Scrolls games.
The objective of this project is to collect reagent-effect data from across the series,
standardize effect naming under a single taxonomy, and create a structured dataset
suitable for fast lookup.

## Key Objectives

- Scrape reagent and effect data from TES alchemy sources.
- Preserve reagent-effect relationships during collection.
- Create a unified naming scheme for equivalent effects.
- Build a normalized relational database.
- Validate data integrity throughout the pipeline.
- Demonstrate practical data acquisition, transformation, and modeling skills.

## Tech Stack

- Python
- BeautifulSoup
- Pandas
- SQLite
- JSON

# 2. Project Architecture

## Pipeline Overview

In [ ]:
Raw TES Sources
        ↓
Web Scraping
        ↓
Intermediate JSON Storage
        ↓
Effect Standardization
        ↓
Final JSON Storage
        ↓
Normalized Relational Database
        ↓
Validation Checks
        ↓
Lookup Dataset

## ERD

The dataset models a many-to-many relationship between reagents and effects. A reagent can have multiple effects, and an effect can appear on multiple reagents. To eliminate redundancy and maintain referential integrity, the data is normalized into two entity tables (`Reagents`, `Effects`) and a junction table (`Reagent_Effects`).

In [ ]:
┌──────────────────────┐
│      Reagents        │
├──────────────────────┤
│ PK id                │
│ name (UNIQUE)        │
│ description          │
└─────────┬────────────┘
          │ 1
          │
          │ N
┌─────────▼────────────┐
│   Reagent_Effects    │
├──────────────────────┤
│ PK/FK reagent_id     │
│ PK/FK effect_id      │
└─────────▲────────────┘
          │ N
          │
          │ 1
┌─────────┴────────────┐
│       Effects        │
├──────────────────────┤
│ PK id                │
│ name (UNIQUE)        │
│ polarity             │
└──────────────────────┘

**New version vs old version comparison:**
* Retrieves names of reagents and of their effects, reagents description (if available) and effects polarity **vs** Retrieving only the names.
* Stores intermediate and final data into dictionary-structured .json files **vs** Plain .txt files output.
* Creates an SQL many-to-many relational database to allow advanced filtering through final data **vs** No data structuring.
* Increased focus on preserving data integrity **vs** No proper integrity check.

**Pipeline steps:**
1. `script1_scrape.py` - Connects to the website, retrieves the data, saves into `01_alchemy_scraped.json`.
2. `script2_transform.py` - Reads `01_alchemy_scraped.json`, replaces effects names, saves into `02_alchemy_transformed.json`.
3. `script3_create_db.py` - Creates `03_alchemy_dataset.sqlite` with empty tables.
4. `script4_insert_data.py` - Reads `02_alchemy_transformed.json`, inserts the data into `03_alchemy_dataset.sqlite`.

**Note on Execution Context:**

The original scripts write results directly to files.  
In this notebook, file I/O has been replaced with in-memory structures and printed output for immediate demonstration.

## Folder Structure

In [ ]:
project_root/
│ main.py                  # Runs the pipeline
│ README.md
│ data/                    # Stores intermediate and final .json files, mapping dictionary .json
|   ├ 01_alchemy_scraped.json
│   ├ 02_alchemy_transformed.json
│   ├ 03_alchemy_dataset.sqlite
│   └ mapping_dic.json
│ scripts/
│   ├ script1_scrape.py
│   ├ script2_transform.py
│   ├ script3_create_db.py
│   └ script4_insert_data.py
│ notebooks/
│   └ tes_alchemy_parser_2.0_overview.ipynb

# 3. Detailed Pipeline

## Step 1 - Data Scraping

The first stage collects reagent and effect information from TES alchemy sources.

**Script:** `script1_scrape.py`

- Scrapes reagent data (name, description) + effect data (name, polarity) per reagent from each of the UESP Alchemy pages (A–Z).
- Stores retrieved data into a dictionary preserving the reagent-effect relationships.
- Saves the dictionary into an intermediate .json file for further processing.

**Code Snippet:**

In [ ]:
import requests
from bs4 import BeautifulSoup as bs
import re
import json

dic = {}

r = requests.get(url)
r.raise_for_status()

soup = bs(r.content, "html.parser")

class_names = re.compile(r"mw-headline|Effect(Pos|Neg|Mix|Other)")


# Filter for specific data:
tags = soup.find_all(
    lambda tag: (  
        any(class_names.search(cls) for cls in tag.get("class", [])) or  # Reagents and effects
        tag.name == "p"  # Descriptions
    )
)

### Why JSON Was Used

The scraped data is first stored as JSON rather than being inserted directly into
a database.

Reasons:

1. Preserve reagent-effect relationships immediately after scraping.
2. Create a reusable intermediate dataset.
3. Simplify debugging and validation.
4. Decouple scraping from transformation logic.

**Code Snippet:**

In [ ]:
dic = {}

if "mw-headline" in tag['class']:       
    current_reagent = item
    dic[current_reagent] = {}

elif "EffectPos" in tag['class']:
    effects = dic[current_reagent].get("effects")
    if not effects:
        dic[current_reagent]["effects"] = {} 
    if "positive" not in dic[current_reagent]["effects"]:
        dic[current_reagent]["effects"]["positive"] = []   
    if text not in dic[current_reagent]["effects"]["positive"]:
        dic[current_reagent]["effects"]["positive"].append(item)

with open("data/01_alchemy_scraped.json", "w") as file:
    json.dump(reagent_data, file, indent=4)

<details>
<summary><strong>Example JSON</strong></summary>

```
{
    "Columbine": {
        "effects": {
            "positive": [
                "Restore Personality",
                "Resist Frost",
                "Fortify Magicka",
                "Chameleon",
                "Restore Health",
                "Restore Magicka",
                "Restore Stamina",
                "Unstoppable"
            ]
        },
        "description": "Columbine is a small plant with many drooping pink flowers that can be found all across Tamriel. It is a prized ingredient for many potions due to its effects."
    }
}
```

### Duplicate Prevention:

The source HTML has hidden effect name duplicates in its structure.

Duplicate effect entries were prevented during scraping.

In [ ]:
if text not in dic[current_reagent]["effects"]["positive"]:
    dic[current_reagent]["effects"]["positive"].append(item)

## Step 2 - Data Standardization

Different TES titles often use different names for mechanically equivalent effects.

To support reliable lookup and future analysis, a unified naming scheme was created.

**Examples:**

In [ ]:
"Heal Health" → "Health Restoration"
"Restore Health" → "Health Restoration"

"Gradual Ravage Health" → "Lingering Health Damage"
"Lingering Damage Health" → "Lingering Health Damage"

"Fortify Health" → "Increased Health"
"Increase Max Health" → "Increased Health"
"Increases health" → "Increased Health"

etc.

**Script:** `script2_transform.py`

- Loads the intermediate .json file with retrieved data.
- Maps effect names into a consistent taxonomy while preserving reagent-effect relation.
- Saves the result into the final .json file.

In [ ]:
import json

raw_data_sample = {
"Columbine": {
        "effects": {
            "positive": [
                "Restore Personality",
                "Resist Frost",
                "Fortify Magicka",
            ]
        }
    }
}
mapping_dic_sample = {
    "Restore Personality": "Personality Restoration",
    "Resist Frost": "Frost Resistance",
    "Fortify Magicka": "Increased Magicka",
}

for key in raw_data_sample:
    effects = raw_data_sample[key].get("effects")
    
    if effects:
        for polarity, effect_list in effects.items():       
            new_effect_list = []       
            
            for effect in effect_list:
                final_name = mapping_dic_sample.get(effect, effect)
                if final_name not in new_effect_list:
                    new_effect_list.append(final_name)
            raw_data_sample[key]['effects'][polarity] = new_effect_list


print(json.dumps(raw_data_sample, indent=4))

'''
Expected output:
{
    "Columbine": {
        "effects": {
            "positive": [
                "Personality Restoration",
                "Frost Resistance",
                "Increased Magicka"
            ]
        }
    }
}
'''

### Data Loss Prevention:

If the website introduces new effect names and the mapping dictionary is not updated, the code will **not** drop the unmapped effects but will preserve the original names instead.

In [ ]:
final_name = mapping_dic.get(effect, effect)

### Duplicate Prevention:

Multiple source effects may map to the same standardized category.

Duplicate entries are prevented during transformation to maintain clean effect lists.

In [ ]:
if final_name not in new_effect_list:
    new_effect_list.append(final_name)

## Intermediate Results Summary

* 503 reagents collected

* 231 unique raw effect names identified

* 104 standardized effect names produced

* Standardized dataset exported to JSON

* Dataset ready for relational modeling and lookup operations

## Step 3 - Database Design

A normalized relational structure was used to avoid duplication and support many-to-many relationships.

- A reagent can possess multiple effects.
- An effect can appear on multiple reagents.

This relationship is represented through a junction table.

**ERD:**

In [ ]:
Reagents
──────────────
PK id
name (UNIQUE)m
description

Effects
──────────────
PK id
name (UNIQUE)
polarity

Reagent_Effects
──────────────
PK/FK reagent_id
PK/FK effect_id


Relationship:
Reagents >───< Reagent_Effects >───< Effects

Many-to-many:
• One reagent may have multiple effects.
• One effect may be shared by multiple reagents.

**Script:** `script3_create_db.py`

- Creates empty database for further data insertion.

**Code Snippet:**

In [1]:
CREATE TABLE Reagents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE,
    description TEXT
);

CREATE TABLE Effects (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT UNIQUE,
    polarity TEXT
);

CREATE TABLE Reagent_Effects (
    reagent_id INTEGER,
    effect_id INTEGER,
    PRIMARY KEY (reagent_id, effect_id),
            
    FOREIGN KEY (reagent_id)
    REFERENCES Reagents(id),

    FOREIGN KEY (effect_id)
    REFERENCES Effects(id)
);

## Step 4 - Database Population

The transformed JSON dataset is inserted into the created database.

Effects are inserted once and linked to reagents through a junction table.

**Script:** `script4_insert_data.py`

- Loads the final .json file with transformed data.
- Inserts the data into corresponding tables with uniqueness constraint.

**Code Snippet:**

In [2]:
cur.execute('''
    INSERT INTO Reagents (name, description) VALUES (?, ?)''',
    (key, description)
)
cur.execute('''
    INSERT OR IGNORE INTO Effects (name, polarity) VALUES (?, ?)''',
    (effect, polarity)
)
                
cur.execute('''
    INSERT OR IGNORE INTO Reagent_Effects
    (reagent_id, effect_id)
    VALUES (?, ?)
    ''',
    (reagent_id, effect_id)
)

# 5. Integrity Validation

## Data Integrity Checks

The transformation stage standardizes effect names across multiple Elder Scrolls titles.

To ensure that no information was unintentionally lost, several validation checks were performed before and after transformation.

### Check 1: Reagent Preservation

The transformation process should modify effect names only.

No reagent records should be added or removed.

In [11]:
import json

with open("../data/01_alchemy_scraped.json", "r") as raw_data:
    raw_dic = json.load(raw_data)

with open("../data/02_alchemy_transformed.json", "r") as transformed_data:
    transformed_dic = json.load(transformed_data)

raw_reagents = len(raw_dic)
transformed_reagents = len(transformed_dic)

print(f"Raw reagents: {raw_reagents}")
print(f"Transformed reagents: {transformed_reagents}")

Raw reagents: 503
Transformed reagents: 503


In [ ]:
Raw reagents: 503
Transformed reagents: 503

All reagent records were preserved during transformation.

### Check 2: Reagent Name Consistency

Even if reagent counts match, individual reagent records could still be lost or renamed accidentally.

In [12]:
raw_names = set(raw_dic.keys())
transformed_names = set(transformed_dic.keys())

missing_reagents = raw_names - transformed_names

print(
    f"Missing reagents: "
    f"{len(missing_reagents)}"
)

Missing reagents: 0


In [ ]:
Missing reagents: 0

All original reagent names were preserved.

### Check 3: Effect Standardization Coverage & Mapping Compression

The objective of the project is to consolidate synonymous effects under a unified naming scheme.

The number of unique effects should decrease after standardization.

In [13]:
raw_effects = set()

for reagent in raw_dic.values():
    effects = reagent.get("effects", {})
    for effect_list in effects.values():
        raw_effects.update(effect_list)

print(
    f"Unique raw effects: "
    f"{len(raw_effects)}"
)

Unique raw effects: 231


In [15]:
with open("../data/mapping_dic.json") as mapping_data:
    mapping_dic = json.load(mapping_data)

print(
    f"Mapping dictionary coverage: "
    f"{len(mapping_dic)}"
)

Mapping dictionary coverage: 222


In [16]:
final_effects = set()

for reagent in transformed_dic.values():

    effects = reagent.get("effects", {})

    for effect_list in effects.values():
        final_effects.update(effect_list)

print(
    f"Total unique standardized effects: "
    f"{len(final_effects)}"
)

Total unique standardized effects: 104


In [ ]:
Unique raw effects: 231
Mapping dictionary coverage: 222
Total unique standardized effects: 104

Effect normalization reduced 231 source effect names to 104 standardized categories.

### Check 4: Unmapped Effects Review

Effects not included in the mapping dictionary should remain unchanged rather than being discarded.

In [17]:
unmapped_effects = []

for effect in raw_effects:

    if effect not in mapping_dic:
        unmapped_effects.append(effect)


for effect in sorted(unmapped_effects):
    print(" ", effect)

print(
    f"\nUnmapped effects: "
    f"{len(unmapped_effects)}"
)

  Attack nearby enemies
  Cannibal Consumption
  Climbing
  Cure Magic
  Damage
  Greenmote Rapture
  Jyggalag's Favor
  Poison Fruit
  Rat Poison

Unmapped effects: 9


In [ ]:
Unmapped effects: 9

Unmapped effects were retained in the final dataset and flagged for manual review.

## SQL Integrity Checks

After loading the transformed dataset into SQLite, relational integrity checks were performed.

#### Check 1: Reagent count

In [18]:
import sqlite3
conn = sqlite3.connect('../data/03_alchemy_dataset.sqlite')
cur = conn.cursor()

In [19]:
cur.execute("SELECT COUNT(*) FROM Reagents")
print(cur.fetchall())

[(503,)]


#### Check 2: Effect count

In [20]:
cur.execute("SELECT COUNT(*) FROM Effects")
print(cur.fetchall())

[(104,)]


#### Check 3: Relationship count

In [7]:
cur.execute("SELECT COUNT(*) FROM Reagent_Effects")
print(cur.fetchall())

[(1852,)]


#### Check 4: Orphan reagent references

In [8]:
cur.execute('''SELECT * FROM Reagent_Effects re
LEFT JOIN Reagents r
ON re.reagent_id = r.id
WHERE r.id IS NULL''')

print(cur.fetchall())

[]


#### Check 5: Orphan effect references

In [9]:
cur.execute('''SELECT * FROM Reagent_Effects re
LEFT JOIN Effects e
ON re.effect_id = e.id
WHERE e.id IS NULL''')

print(cur.fetchall())

[]


#### Check 6: Join Validation (with Pandas)

In [22]:
import pandas as pd

query = """
SELECT
    Reagents.name AS reagent_name,
    Effects.name AS effect_name,
    Effects.polarity,
    Reagents.description
FROM Reagents
JOIN Reagent_Effects
    ON Reagent_Effects.reagent_id = Reagents.id
JOIN Effects
    ON Reagent_Effects.effect_id = Effects.id
WHERE Reagents.id > 93 
    AND Reagents.id < 97
"""

df = pd.read_sql(query, conn)

df.set_index(['reagent_name', 'polarity']).sort_index(ascending=[True, False])

effect_name  \
reagent_name       polarity                             
Clouded Funnel Cap positive  Intelligence Restoration   
                   positive     Intelligence Increase   
                   negative            Magicka Damage   
                   negative        Endurance Decrease   
Clover             positive             Luck Increase   
Coda Flower        positive         Strength Increase   
                   positive                Levitation   
                   negative     Intelligence Decrease   
                   negative            Magicka Damage   
                   negative  Lingering Stamina Damage   
                   negative             Health Damage   
                   negative      Personality Decrease   

                                                                   description  
reagent_name       polarity                                                     
Clouded Funnel Cap positive  Clouded Funnel Caps comes from the fungus Clou...  
                   positive  Clouded Funnel Caps comes from the fungus Clou...  
                   negative  Clouded Funnel Caps comes from the fungus Clou...  
                   negative  Clouded Funnel Caps comes from the fungus Clou...  
Clover             positive                                                NaN  
Coda Flower        positive  The coda flower is the fruiting body collected...  
                   positive  The coda flower is the fruiting body collected...  
                   negative  The coda flower is the fruiting body collected...  
                   negative  The coda flower is the fruiting body collected...  
                   negative  The coda flower is the fruiting body collected...  
                   negative  The coda flower is the fruiting body collected...  
                   negative  The coda flower is the fruiting body collected...

## Validation Summary

In [ ]:
Reagents before transformation: 503
Reagents after transformation: 503

Unique effect names before transformation: 231
Unique effect names after transformation: 104

Source effect names covered by the mapping dictionary: 222
Effects not included in the mapping dictionary: 9

Database integrity checks found no orphan records.

### Conclusion

* The results indicate that the normalization process successfully standardized effect names while preserving the data integrity.

* The number of unique effects has expectedly decreased after standardization.

* Unmapped effects were retained in the final dataset and flagged for manual review.

# 6. Lookup Demonstration

SQL dataset allows rapid lookup and complex queries.

Example questions:

- What effects are associated with a given reagent(s)?
- Which reagents have a given effect(s)?
- Which reagents can be found in a given zone?
- Which reagents share a given effect and are located in a given zone?

etc.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/03_alchemy_dataset.sqlite')
cur = conn.cursor()

### Lookup Example 1: What effects do Jarrin Root and Vampire Dust have?

In [3]:
query = """
    SELECT
        R.name AS reagent_name,
        E.polarity,
        E.name AS effect_name
    FROM Effects E
    JOIN Reagent_Effects RE
        ON E.id = RE.effect_id
    JOIN Reagents R
        ON R.id = RE.reagent_id
    WHERE R.name IN ("Jarrin Root", "Vampire Dust")
    ORDER BY E.name
    """

pd.read_sql(query, conn).set_index(["reagent_name", "polarity"]).sort_index(ascending=True)

effect_name
reagent_name polarity                          
Jarrin Root  negative            Focus Decrease
             negative             Health Damage
             negative            Magicka Damage
             negative            Stamina Damage
Vampire Dust negative  Frost Spells Enhancement
             negative                   Silence
             negative                 Vampirism
             positive              Disease Cure
             positive        Disease Resistance
             positive          Increased Health
             positive              Invisibility
             positive       Magicka Restoration
             positive         Strength Increase
             positive         Vitality Increase
             positive        Willpower Increase

### Lookup Example 2: Which reagents have Silence effect?

In [4]:
query = """
SELECT r.name, r.description
FROM Reagents r
JOIN Reagent_Effects re
    ON r.id = re.reagent_id
JOIN Effects e
    ON e.id = re.effect_id
WHERE e.name = 'Silence'
ORDER BY r.name
"""

pd.set_option('display.max_colwidth', None)
pd.read_sql(query, conn).set_index(['name'])

,description
name,
Ashen Remains,"Ashen Remains are the remnants of bones burned using the Crematorium in Ebrocca, in the Shivering Isles."
Bergamot Seeds,"Bergamot Seeds comes from the plant Bergamot, most commonly found in the Great Forest region of Cyrodiil."
Black rose,NaN
Daedra Heart,"A Daedra Heart (or aortas daedrae) is an ingredient that can be harvested from slain Daedra, primarily Dremora. In older times, they could be found on many other varieties of Daedra, including Clannfears, Daedroths, Golden Saints, Hungers, Ogrims, and Xivilai. These beings, to the extent that they can still be encountered, now produce other ingredients. But Dremora have consistently been a reliable source for Daedra Hearts."
Elytra Ichor,Elytra Ichor is harvested from dead elytra which are native to the Shivering Isles.
Frost Salts,Frost Salts are the crystalline compound that may be collected from the remains of frost atronachs that have been banished from the mortal plane. Refined Frost Salts are samples prepared by mages possessing even greater purity.
Grummite Eggs,NaN
Harrada,"Harrada comes from the Harrada Root plant, which is native to Mehrunes Dagon's realm of Oblivion, the Deadlands. Normally Harrada Root plants are not found in Tamriel, however they take root outside of certain Oblivion Gates that lead to the Deadlands."
Human Heart,NaN


### Lookup Example 3: Which reagents have Silence effect and can also be found on Shivering Isles?

In [5]:
query = """
SELECT
    R.name,
    R.description
FROM Reagents r
JOIN Reagent_Effects re
    ON re.reagent_id = r.id
JOIN Effects e
    ON re.effect_id = e.id
WHERE 
    e.name = 'Silence' AND
    description LIKE '%Shivering Isles%'
ORDER BY R.name
"""

pd.read_sql(query, conn).set_index(['name'])

,description
name,
Ashen Remains,"Ashen Remains are the remnants of bones burned using the Crematorium in Ebrocca, in the Shivering Isles."
Elytra Ichor,Elytra Ichor is harvested from dead elytra which are native to the Shivering Isles.


### Function Implementation

Alternatively, a function (with or without input prompt) could be designed individually for any given query, for example:

In [8]:
def give_effects_per_reagent ():

    reagents = input("Enter reagent name(s) separated by commas: ")

    if isinstance(reagents, str):
        if ',' in reagents:
            reagents = [reagent.strip().title() for reagent in reagents.split(',')]
        else:
            reagents = [reagents.title()]


    placeholders = ', '.join(['?'] * len(reagents))
    
    query = f"""
    SELECT
        R.name AS reagent_name,
        E.polarity,
        E.name AS effect_name
    FROM Effects E
    JOIN Reagent_Effects RE
        ON E.id = RE.effect_id
    JOIN Reagents R
        ON R.id = RE.reagent_id
    WHERE R.name IN ({placeholders})
    ORDER BY E.name
    """

    return pd.read_sql(query, conn, params=reagents).set_index(["reagent_name", "polarity"]).sort_index(ascending=True)

give_effects_per_reagent()

effect_name
reagent_name polarity                      
Bread        negative      Agility Decrease
             negative     Strength Decrease
             positive   Fatigue Restoration
             positive    Health Restoration
             positive   Perception Increase
Cheese Wheel negative           Luck Damage
             positive   Fatigue Restoration
             positive    Health Restoration
             positive  Paralysis Resistance
             positive    Willpower Increase

In [32]:
conn.close()

# 7. Challenges and Lessons Learned

## Challenges

- Inconsistent effect naming between games.
- Preserving relationships during scraping.
- Designing a reusable intermediate format.
- Maintaining data integrity during transformation.

## Lessons Learned

- Data modeling decisions strongly affect downstream usability.
- Validation is as important as collection.
- Intermediate data formats simplify pipeline development.
- Normalization reduces redundancy and improves maintainability.